# 01 — Shared Preprocessing

**Owner:** Khushi (ONLY Khushi edits this notebook)

---

### Purpose
- Load all raw datasets from `data/raw/`
- Clean, merge, and feature-engineer
- Export a single `final_dataset.csv` to `data/processed/`

### Inputs
| File | Path |
|------|------|
| Production data | `data/raw/production.csv` |
| Water quality | `data/raw/water_quality.csv` |
| Climate data | `data/raw/climate.csv` |
| Genomic data | `data/raw/genomic.csv` |

### Output
| File | Path |
|------|------|
| Final merged dataset | `data/processed/final_dataset.csv` |

> **Rule:** Everyone reads `final_dataset.csv`. Nobody modifies it manually.

In [1]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import os

In [2]:
# ============================================================
# LOAD RAW DATASETS
# ============================================================
# Adjust paths if running from a different directory
DATA_RAW = os.path.join("..", "data", "raw")

# production = pd.read_csv(os.path.join(DATA_RAW, "production.csv"))
# water      = pd.read_csv(os.path.join(DATA_RAW, "water_quality.csv"))
# climate    = pd.read_csv(os.path.join(DATA_RAW, "climate.csv"))
# genomic    = pd.read_csv(os.path.join(DATA_RAW, "genomic.csv"))

print("TODO: Uncomment above lines after placing real CSVs in data/raw/")

TODO: Uncomment above lines after placing real CSVs in data/raw/


In [3]:
# ============================================================
# DATA CLEANING & MERGING
# ============================================================
# TODO: Handle missing values, merge datasets on common keys,
#       encode categorical variables, normalize if needed.

# Example:
# merged = production.merge(water, on='sample_id', how='inner')
# merged = merged.merge(climate, on=['date', 'region'], how='left')
# merged = merged.merge(genomic, on='sample_id', how='left')

print("TODO: Implement cleaning and merging logic")

TODO: Implement cleaning and merging logic


In [4]:
# ============================================================
# EXPORT FINAL DATASET
# ============================================================
OUTPUT_PATH = os.path.join("..", "data", "processed", "final_dataset.csv")

# merged.to_csv(OUTPUT_PATH, index=False)
# print(f"Saved final_dataset.csv → {OUTPUT_PATH}")
# print(f"Shape: {merged.shape}")

print("TODO: Uncomment above after merging is complete")

TODO: Uncomment above after merging is complete


## Shared Disease Evaluation Pipeline (Step 1 + Step 2)

This section finalizes a trainable disease-prediction target and defines a common split/metric/timing pipeline for all members.

- Target used for modeling: `disease_risk_target` (derived from time trend, production shock, and environment/genomic stress)
- Shared split rule: stratified 80/20 with `random_state=42`
- Shared metrics: Accuracy, Precision (macro), Recall (macro), F1 (macro)
- Shared timing: train seconds and inference milliseconds per 1000 rows
- Shared resource metrics: model size (MB) and peak training RAM (MB)
- Structured text only: no image outputs

In [5]:
from pathlib import Path
import json
import time
import tempfile
import tracemalloc

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("..") / "data" / "processed" / "final_dataset.csv"
OUTPUT_DIR = Path("..") / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COL = "disease_risk_target"


def build_disease_target(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()

    if "country" in work.columns and "year" in work.columns:
        work = work.sort_values(["country", "year"]).reset_index(drop=True)
    elif "year" in work.columns:
        work = work.sort_values(["year"]).reset_index(drop=True)

    # Production shock: year-over-year drop by country (or global fallback)
    if "country" in work.columns and "production" in work.columns:
        pct_change = work.groupby("country", dropna=False)["production"].pct_change()
    elif "production" in work.columns:
        pct_change = work["production"].pct_change()
    else:
        pct_change = pd.Series(np.zeros(len(work)), index=work.index)

    production_drop = np.maximum(0.0, -pct_change.fillna(0.0))

    env_cols = [
        c for c in [
            "temperature_celsius",
            "precip_mm",
            "humidity",
            "water_Salinity (ppt)",
            "water_pH",
            "water_WaterTemp (C)",
            "water_AirTemp (C)",
            "water_SecchiDepth (m)",
            "water_WaterDepth (m)",
        ]
        if c in work.columns
    ]

    genomic_cols = [
        c for c in [
            "genomic_Mutation_Flag_global_mean",
            "genomic_kmer_3_freq_global_mean",
            "genomic_GC_Content_global_mean",
            "genomic_AT_Content_global_mean",
        ]
        if c in work.columns
    ]

    env_stress = work[env_cols].abs().mean(axis=1) if env_cols else pd.Series(0.0, index=work.index)
    genomic_instability = (
        work[genomic_cols].abs().mean(axis=1) if genomic_cols else pd.Series(0.0, index=work.index)
    )

    risk_score = 0.50 * production_drop + 0.35 * env_stress + 0.15 * genomic_instability

    q1 = float(risk_score.quantile(0.50))
    q2 = float(risk_score.quantile(0.80))

    labels = np.where(risk_score <= q1, "Low", np.where(risk_score <= q2, "Medium", "High"))
    work[TARGET_COL] = labels

    # Keep the original score for analysis and debugging.
    work["disease_risk_score"] = risk_score

    return work


def build_feature_matrix(df: pd.DataFrame, target_col: str):
    drop_cols = {target_col, "disease_risk_score"}
    if "genomic_Disease_Risk_global_mode" in df.columns:
        drop_cols.add("genomic_Disease_Risk_global_mode")

    feature_cols = [c for c in df.columns if c not in drop_cols]
    X = df[feature_cols].copy()
    y = df[target_col].copy()

    categorical_cols = [c for c in X.columns if X[c].dtype == "object"]
    numeric_cols = [c for c in X.columns if c not in categorical_cols]

    return X, y, numeric_cols, categorical_cols


def make_preprocessor(numeric_cols, categorical_cols):
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ],
        remainder="drop",
    )


df = pd.read_csv(DATA_PATH)
df = build_disease_target(df)
X, y, numeric_cols, categorical_cols = build_feature_matrix(df, TARGET_COL)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_meta = {
    "target_column": TARGET_COL,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "class_distribution": y.value_counts().to_dict(),
    "train_shape": [int(X_train.shape[0]), int(X_train.shape[1])],
    "test_shape": [int(X_test.shape[0]), int(X_test.shape[1])],
}

(OUTPUT_DIR / "shared_split_metadata.json").write_text(json.dumps(split_meta, indent=2), encoding="utf-8")
df.to_csv(OUTPUT_DIR / "final_dataset_with_disease_target.csv", index=False)

print("Shared target and split prepared.")
print(json.dumps(split_meta, indent=2))

Shared target and split prepared.
{
  "target_column": "disease_risk_target",
  "test_size": 0.2,
  "random_state": 42,
  "class_distribution": {
    "Low": 5888,
    "Medium": 3555,
    "High": 2214
  },
  "train_shape": [
    9325,
    21
  ],
  "test_shape": [
    2332,
    21
  ]
}


In [6]:
def evaluate_classifier(model_name: str, estimator, X_train, X_test, y_train, y_test, preprocessor):
    pipeline = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("model", estimator),
        ]
    )

    tracemalloc.start()
    t0 = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_seconds = time.perf_counter() - t0
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    t1 = time.perf_counter()
    y_pred = pipeline.predict(X_test)
    infer_seconds = time.perf_counter() - t1

    n_test = max(len(X_test), 1)
    infer_ms_per_1000 = (infer_seconds / n_test) * 1000.0 * 1000.0

    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=True) as tmp:
        joblib.dump(pipeline, tmp.name)
        model_size_mb = Path(tmp.name).stat().st_size / (1024 * 1024)

    row = {
        "model": model_name,
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision_macro": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
        "train_seconds": float(train_seconds),
        "infer_ms_per_1000": float(infer_ms_per_1000),
        "model_size_mb": float(model_size_mb),
        "peak_train_ram_mb": float(peak_bytes / (1024 * 1024)),
    }

    return row, pipeline


print("Shared evaluator ready.")
print("Use evaluate_classifier(...) in each member notebook with the same split and metric conventions.")

Shared evaluator ready.
Use evaluate_classifier(...) in each member notebook with the same split and metric conventions.
